# ACPT / AllCPT (Polish)

Computes two made-up metrics over BM25 and BGE-M3 dense (cw=8192, bs=32) for 22 selected queries on the `polish_final_cluster_reparsed/merged` augmented chunks corpus.

- **ACPT (Any Context Precedes Target):** `True` if any `utilized_context_chunk_ids` rank is strictly better (lower) than the target rank.
- **AllCPT (All Contexts Precede Target):** `True` if every utilized-context rank is strictly better than the target rank.

Ranks beyond top-100 (or missing) are treated as `inf`; `inf < inf` is `False`.

In [14]:
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from lcr.formatter import DataFormatter
from lcr.modeling.bm25_embedder import BM25Embedder
from lcr.modeling.bge_m3_embedder import BGEM3Embedder

DATA_DIR = Path("../data/processed/polish_final_cluster_reparsed/merged")
CHUNKS_PATH = DATA_DIR / "augmented_chunks.jsonl"
QUERIES_PATH = DATA_DIR / "contextual_queries.jsonl"

CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
BGE_CACHE = CACHE_DIR / "bge_m3_dense_augmented_chunks_polish.npz"

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print("device:", DEVICE)

TARGET_QIDS = [
    "financingeducation_688",
    "financingeducation_653",
    "financingeducation_652",
    "financingeducation_960",
    "healthcarepublicfunds_638",
    "healthcarepublicfunds_743",
    "obligationtodefend_2223",
    "obligationtodefend_1846",
    "police_1249",
    "police_368",
    "police_1271",
    "socialassistance_242",
    "socialassistance_361",
    "socialassistance_1179",
    "socialassistance_1603",
    "socialinsurancesystem_1052",
    "socialinsurancesystem_622",
    "socialinsurancesystem_367",
    "socialinsurancesystem_1815",
    "statefireservice_1636",
    "statefireservice_1711",
    "statefireservice_594",
]
print("target queries:", len(TARGET_QIDS))

device: mps
target queries: 22


## Load + filter data

In [2]:
formatter = DataFormatter()
formatter.load_from_jsonl(CHUNKS_PATH, "documents")
formatter.load_from_jsonl(QUERIES_PATH, "queries")

target_set = set(TARGET_QIDS)
formatter.queries_dataset = formatter.queries_dataset.filter(lambda x: x["chunk_id"] in target_set)
print("filtered queries:", len(formatter.queries_dataset))
assert len(formatter.queries_dataset) == len(TARGET_QIDS), (
    f"expected {len(TARGET_QIDS)} queries, got {len(formatter.queries_dataset)}"
)

utilized_by_qid: dict[str, list[str]] = {
    row["chunk_id"]: list(row["utilized_context_chunk_ids"] or [])
    for row in formatter.queries_dataset
}
for qid in TARGET_QIDS:
    print(f"  {qid}: {len(utilized_by_qid[qid])} utilized context(s)")

Generating train split: 15967 examples [00:00, 632757.79 examples/s]


Loaded documents dataset from jsonl ../data/processed/polish_final_cluster_reparsed/merged/augmented_chunks.jsonl


Map: 100%|██████████| 15967/15967 [00:00<00:00, 97524.48 examples/s]
Generating train split: 300 examples [00:00, 135869.91 examples/s]


Loaded queries dataset from jsonl ../data/processed/polish_final_cluster_reparsed/merged/contextual_queries.jsonl


Filter: 100%|██████████| 300/300 [00:00<00:00, 38549.41 examples/s]

filtered queries: 22
  financingeducation_688: 1 utilized context(s)
  financingeducation_653: 1 utilized context(s)
  financingeducation_652: 1 utilized context(s)
  financingeducation_960: 7 utilized context(s)
  healthcarepublicfunds_638: 1 utilized context(s)
  healthcarepublicfunds_743: 2 utilized context(s)
  obligationtodefend_2223: 2 utilized context(s)
  obligationtodefend_1846: 2 utilized context(s)
  police_1249: 1 utilized context(s)
  police_368: 3 utilized context(s)
  police_1271: 5 utilized context(s)
  socialassistance_242: 3 utilized context(s)
  socialassistance_361: 2 utilized context(s)
  socialassistance_1179: 1 utilized context(s)
  socialassistance_1603: 2 utilized context(s)
  socialinsurancesystem_1052: 1 utilized context(s)
  socialinsurancesystem_622: 2 utilized context(s)
  socialinsurancesystem_367: 3 utilized context(s)
  socialinsurancesystem_1815: 1 utilized context(s)
  statefireservice_1636: 2 utilized context(s)
  statefireservice_1711: 1 utilized co

## BM25 ranking

In [3]:
bm25 = BM25Embedder(spacy_model="pl_core_news_sm")
t0 = time.time()
bm25_results, bm25_label_ids, _ = bm25.compute_results(formatter)
print(f"BM25 done in {time.time() - t0:.1f}s. queries: {len(bm25_label_ids)}, top-N per query: {len(next(iter(bm25_results.values())))}")

BM25 done in 220.5s. queries: 22, top-N per query: 100


## BGE-M3 dense ranking (cached doc embeddings)

In [4]:
bge = BGEM3Embedder(encoding_type="dense", batch_size=32, device=DEVICE)

chunks, chunk_ids = formatter.get_flattened()

if BGE_CACHE.exists():
    t0 = time.time()
    cache = np.load(BGE_CACHE, allow_pickle=False)
    doc_emb = cache["embeddings"]
    cached_ids = cache["chunk_ids"].tolist()
    assert cached_ids == list(chunk_ids), "cache chunk_ids don't match current corpus order; delete cache to rebuild"
    print(f"loaded BGE-M3 doc embeddings from cache in {time.time() - t0:.1f}s shape={doc_emb.shape}")
else:
    t0 = time.time()
    doc_emb = bge.embed_documents(chunks)
    doc_emb = np.asarray(doc_emb, dtype=np.float32)
    np.savez(BGE_CACHE, embeddings=doc_emb, chunk_ids=np.array(chunk_ids))
    print(f"encoded {len(chunks)} docs in {time.time() - t0:.1f}s, cached to {BGE_CACHE} shape={doc_emb.shape}")

pre tokenize: 100%|██████████| 499/499 [00:00<00:00, 763.13it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 499/499 [04:15<00:00,  1.95it/s]


encoded 15967 docs in 263.3s, cached to cache/bge_m3_dense_augmented_chunks_polish.npz shape=(15967, 1024)


In [5]:
queries, bge_label_ids = formatter.get_queries()
t0 = time.time()
q_emb = bge.embed_queries(queries)
q_emb = np.asarray(q_emb, dtype=np.float32)
print(f"encoded {len(queries)} queries in {time.time() - t0:.1f}s shape={q_emb.shape}")

q_t = bge._convert_to_tensor(q_emb)
d_t = bge._convert_to_tensor(doc_emb)
scores = bge.get_similarities(q_t, d_t).numpy()
bge_results = bge.get_results(scores, list(chunk_ids))
print(f"BGE-M3 ranking done. top-N per query: {len(next(iter(bge_results.values())))}")

encoded 22 queries in 0.6s shape=(22, 1024)
BGE-M3 ranking done. top-N per query: 100


## Compute ACPT / AllCPT

In [6]:
def ranks_from_results(per_query_scores: dict[str, float]) -> dict[str, int]:
    items = sorted(per_query_scores.items(), key=lambda kv: kv[1], reverse=True)
    return {cid: r for r, (cid, _) in enumerate(items, start=1)}


def rank_of(rank_map: dict[str, int], cid: str) -> float:
    return rank_map.get(cid, math.inf)


def fmt_rank(r: float) -> str:
    return "inf" if r == math.inf else str(int(r))


def metric_rows(retriever_name: str, results: dict, label_ids: list[str]) -> list[dict]:
    rows = []
    for i, qid in enumerate(label_ids):
        rank_map = ranks_from_results(results[str(i)])
        target_rank = rank_of(rank_map, qid)
        ctx_ids = utilized_by_qid.get(qid, [])
        ctx_ranks = [rank_of(rank_map, c) for c in ctx_ids]
        acpt = any(cr < target_rank for cr in ctx_ranks)
        allcpt = bool(ctx_ranks) and all(cr < target_rank for cr in ctx_ranks)
        rows.append({
            "retriever": retriever_name,
            "query_chunk_id": qid,
            "target_rank": fmt_rank(target_rank),
            "n_contexts": len(ctx_ids),
            "context_ranks": [fmt_rank(cr) for cr in ctx_ranks],
            "ACPT": acpt,
            "AllCPT": allcpt,
        })
    return rows


bm25_rows = metric_rows("bm25", bm25_results, bm25_label_ids)
bge_rows = metric_rows("bge-m3-dense", bge_results, bge_label_ids)
df_bm25 = pd.DataFrame(bm25_rows)
df_bge = pd.DataFrame(bge_rows)

### Per-query: BM25

In [7]:
df_bm25

,retriever,query_chunk_id,target_rank,n_contexts,context_ranks,ACPT,AllCPT
0,bm25,financingeducation_652,24,1,[8],True,True
1,bm25,financingeducation_653,5,1,[58],False,False
2,bm25,financingeducation_688,inf,1,[2],True,True
3,bm25,financingeducation_960,30,7,"[1, 2, 3, 5, inf, inf, inf]",True,False
4,bm25,healthcarepublicfunds_638,inf,1,[1],True,True
5,bm25,healthcarepublicfunds_743,4,2,"[2, 1]",True,True
6,bm25,obligationtodefend_1846,3,2,"[2, 1]",True,True
7,bm25,obligationtodefend_2223,1,2,"[inf, 8]",False,False
8,bm25,police_1249,7,1,[1],True,True
9,bm25,police_1271,46,5,"[1, 6, 10, 33, 3]",True,True


### Per-query: BGE-M3 dense

In [8]:
df_bge

,retriever,query_chunk_id,target_rank,n_contexts,context_ranks,ACPT,AllCPT
0,bge-m3-dense,financingeducation_652,20,1,[3],True,True
1,bge-m3-dense,financingeducation_653,1,1,[37],False,False
2,bge-m3-dense,financingeducation_688,inf,1,[4],True,True
3,bge-m3-dense,financingeducation_960,7,7,"[4, 37, 1, 9, 86, inf, inf]",True,False
4,bge-m3-dense,healthcarepublicfunds_638,inf,1,[1],True,True
5,bge-m3-dense,healthcarepublicfunds_743,4,2,"[3, 1]",True,True
6,bge-m3-dense,obligationtodefend_1846,7,2,"[2, 1]",True,True
7,bge-m3-dense,obligationtodefend_2223,1,2,"[inf, 31]",False,False
8,bge-m3-dense,police_1249,5,1,[2],True,True
9,bge-m3-dense,police_1271,2,5,"[1, 12, 20, 19, 13]",True,False


### Side-by-side

In [9]:
merged = (
    df_bm25.rename(columns={
        "target_rank": "target_rank_bm25",
        "context_ranks": "context_ranks_bm25",
        "ACPT": "ACPT_bm25",
        "AllCPT": "AllCPT_bm25",
    })[["query_chunk_id", "n_contexts", "target_rank_bm25", "context_ranks_bm25", "ACPT_bm25", "AllCPT_bm25"]]
    .merge(
        df_bge.rename(columns={
            "target_rank": "target_rank_bge",
            "context_ranks": "context_ranks_bge",
            "ACPT": "ACPT_bge",
            "AllCPT": "AllCPT_bge",
        })[["query_chunk_id", "target_rank_bge", "context_ranks_bge", "ACPT_bge", "AllCPT_bge"]],
        on="query_chunk_id",
    )
)
merged

,query_chunk_id,n_contexts,target_rank_bm25,context_ranks_bm25,ACPT_bm25,AllCPT_bm25,target_rank_bge,context_ranks_bge,ACPT_bge,AllCPT_bge
0,financingeducation_652,1,24,[8],True,True,20,[3],True,True
1,financingeducation_653,1,5,[58],False,False,1,[37],False,False
2,financingeducation_688,1,inf,[2],True,True,inf,[4],True,True
3,financingeducation_960,7,30,"[1, 2, 3, 5, inf, inf, inf]",True,False,7,"[4, 37, 1, 9, 86, inf, inf]",True,False
4,healthcarepublicfunds_638,1,inf,[1],True,True,inf,[1],True,True
5,healthcarepublicfunds_743,2,4,"[2, 1]",True,True,4,"[3, 1]",True,True
6,obligationtodefend_1846,2,3,"[2, 1]",True,True,7,"[2, 1]",True,True
7,obligationtodefend_2223,2,1,"[inf, 8]",False,False,1,"[inf, 31]",False,False
8,police_1249,1,7,[1],True,True,5,[2],True,True
9,police_1271,5,46,"[1, 6, 10, 33, 3]",True,True,2,"[1, 12, 20, 19, 13]",True,False


### Aggregate

In [10]:
agg = pd.DataFrame([
    {"retriever": "bm25", "ACPT_mean": df_bm25["ACPT"].mean(), "AllCPT_mean": df_bm25["AllCPT"].mean()},
    {"retriever": "bge-m3-dense", "ACPT_mean": df_bge["ACPT"].mean(), "AllCPT_mean": df_bge["AllCPT"].mean()},
])
agg

,retriever,ACPT_mean,AllCPT_mean
0,bm25,0.681818,0.500000
1,bge-m3-dense,0.772727,0.454545


In [12]:
# give counts, insted of means
agg_counts = pd.DataFrame([
    {"retriever": "bm25", "ACPT_count": df_bm25["ACPT"].sum(), "AllCPT_count": df_bm25["AllCPT"].sum()},
    {"retriever": "bge-m3-dense", "ACPT_count": df_bge["ACPT"].sum(), "AllCPT_count": df_bge["AllCPT"].sum()},
])
agg_counts

,retriever,ACPT_count,AllCPT_count
0,bm25,15,11
1,bge-m3-dense,17,10


### Solved counts (target in top-10)

In [ ]:
def _num(r):
    return math.inf if r == "inf" else int(r)

bm25_solved = sum(_num(r) <= 10 for r in merged["target_rank_bm25"])
bge_solved = sum(_num(r) <= 10 for r in merged["target_rank_bge"])
either_solved = sum(
    (_num(a) <= 10) or (_num(b) <= 10)
    for a, b in zip(merged["target_rank_bm25"], merged["target_rank_bge"])
)
n_total = len(merged)
print(f"solved by BM25:           {bm25_solved}/{n_total}")
print(f"solved by BGE-M3 dense:   {bge_solved}/{n_total}")
print(f"solved by at least one:   {either_solved}/{n_total}")